# SDXL Convert B2 TPU UNet

Generated from `kaggle/common/sdxl_tpu_common.py`.


In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME = '__MODEL_NAME__'
REALISTIC = '__REALISTIC__'
MIN_SOC = '__MIN_SOC__'
COMMON_CODE = 'import glob\nimport json\nimport os\nimport re\nimport shutil\nimport signal\nimport select\nimport subprocess\nimport time\n\n\nLOG_DIR = "/kaggle/working/logs"\n\n\ndef now_hms():\n    return time.strftime("%H:%M:%S")\n\n\ndef run(cmd, log=None, check=True, cwd=None, env=None):\n    if log:\n        os.makedirs(os.path.dirname(log), exist_ok=True)\n        wrapped = f"set -eo pipefail; {cmd} 2>&1 | tee {log}"\n    else:\n        wrapped = f"set -eo pipefail; {cmd}"\n    rc = subprocess.call(["bash", "-lc", wrapped], cwd=cwd, env=env)\n    if check and rc != 0:\n        raise RuntimeError(f"shell failed rc={rc}: {cmd[:240]} log={log}")\n    return rc\n\n\ndef file_mib(path):\n    try:\n        return os.path.getsize(path) / (1024 * 1024)\n    except OSError:\n        return 0.0\n\n\ndef mem_avail_mb():\n    try:\n        for line in open("/proc/meminfo"):\n            if line.startswith("MemAvailable:"):\n                return int(line.split()[1]) // 1024\n    except Exception:\n        pass\n    return -1\n\n\ndef disk_avail_mb(path="/kaggle/working"):\n    try:\n        st = os.statvfs(path)\n        return int(st.f_bavail * st.f_frsize / (1024 * 1024))\n    except Exception:\n        return -1\n\n\ndef tsv_escape(value, limit=1400):\n    return str(value).replace("\\t", " ").replace("\\r", " ").replace("\\n", " ")[:limit]\n\n\ndef configure_env(model_name, min_soc, realistic, civitai_version_id=None):\n    os.environ["MODEL_NAME"] = model_name\n    os.environ["MIN_SOC"] = min_soc\n    os.environ["REALISTIC"] = str(realistic).lower()\n    if civitai_version_id is not None:\n        os.environ["CIVITAI_VERSION_ID"] = str(civitai_version_id)\n    os.makedirs(LOG_DIR, exist_ok=True)\n    print(\n        f"[*] {now_hms()} config model={model_name} soc={min_soc} "\n        f"realistic={os.environ[\'REALISTIC\']}"\n    )\n\n\ndef install_tools_and_convert_package():\n    print(f"[*] {now_hms()} installing system tools and convertsdxl")\n    run("apt-get update -y > /dev/null")\n    run("apt-get install -y unzip zip curl time libc++1-19 libc++abi1-19 libunwind-19 > /dev/null")\n    run("ldconfig")\n    run("curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1")\n    os.environ["PATH"] = f"/root/.local/bin:{os.environ[\'PATH\']}"\n    run("rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip")\n    run(\n        "curl -sL --fail --retry 5 --retry-delay 5 "\n        "-o /tmp/convertsdxl.zip \'https://chino.icu/local-dream/convertsdxl.zip\'"\n    )\n    run("mkdir -p /tmp/convertsdxl_unzip")\n    run("unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip")\n    root = subprocess.check_output(\n        "find /tmp/convertsdxl_unzip -maxdepth 3 -name \'export_sdxl.sh\' -printf \'%h\\n\' | head -1",\n        shell=True,\n        text=True,\n    ).strip()\n    assert root, "export_sdxl.sh not found in convertsdxl.zip"\n    os.environ["NPUCONVERT_DIR"] = os.path.abspath(root)\n    print(f"[*] {now_hms()} NPUCONVERT_DIR={os.environ[\'NPUCONVERT_DIR\']}")\n\n\ndef install_qnn_sdk():\n    print(f"[*] {now_hms()} installing QNN SDK")\n    run("mkdir -p /tmp/qnn_sdk")\n    run(\n        "curl -sL --fail --retry 5 --retry-delay 5 -A \'Mozilla/5.0\' "\n        "-o /tmp/qnn_sdk/qnn.zip "\n        "\'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/"\n        "qualcomm_neural_processing_sdk/v2.28.0.241029.zip\'"\n    )\n    run("cd /tmp/qnn_sdk && unzip -q qnn.zip")\n    envsetup = subprocess.check_output(\n        "find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit",\n        shell=True,\n        text=True,\n    ).strip()\n    assert envsetup, "envsetup.sh not found in QNN SDK"\n    os.environ["QNN_SDK_BIN"] = os.path.dirname(envsetup)\n    os.environ["QNN_SDK_ROOT_DIR"] = os.path.dirname(os.environ["QNN_SDK_BIN"])\n    print(f"[*] {now_hms()} QNN_SDK_ROOT={os.environ[\'QNN_SDK_ROOT_DIR\']}")\n\n\ndef setup_python_env():\n    os.chdir(os.environ["NPUCONVERT_DIR"])\n    run("uv venv -p 3.10.17 --clear")\n    run(". .venv/bin/activate && uv sync")\n\n\ndef locate_phase1_output():\n    pkl = glob.glob("/kaggle/input/**/data_sdxl.pkl", recursive=True)\n    assert pkl, "Phase 1 data_sdxl.pkl not found in /kaggle/input"\n    phase1_dir = os.path.dirname(pkl[0])\n    model = glob.glob("/kaggle/input/**/model.safetensors", recursive=True)\n    assert model, "model.safetensors not found in /kaggle/input"\n    size = os.path.getsize(model[0])\n    assert size > int(1e9), f"model.safetensors truncated: {size}"\n    os.environ["PHASE1_DIR"] = phase1_dir\n    os.environ["MODEL_PATH"] = model[0]\n    print(f"[*] {now_hms()} Phase1 dir={phase1_dir}")\n    print(f"[*] {now_hms()} MODEL_PATH size={size / 1e9:.2f}GB")\n\n\ndef copy_phase1_calibration_data():\n    npu = os.environ["NPUCONVERT_DIR"]\n    pkl = os.path.join(os.environ["PHASE1_DIR"], "data_sdxl.pkl")\n    shutil.copy(pkl, f"{npu}/data_sdxl.pkl")\n    src_img = f"{os.environ[\'PHASE1_DIR\']}/images_sdxl"\n    assert os.path.isdir(src_img), f"{src_img} missing"\n    dst_img = f"{npu}/images_sdxl"\n    if os.path.exists(dst_img):\n        shutil.rmtree(dst_img)\n    shutil.copytree(src_img, dst_img)\n    assert os.listdir(dst_img), f"{dst_img} empty"\n\n\ndef start_background_services():\n    os.makedirs(LOG_DIR, exist_ok=True)\n    watcher_sh = """#!/bin/bash\necho "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"\nwhile true; do\n  epoch=$(date +%s); dt=$(date -Iseconds)\n  mem=$(awk \'/MemAvailable:/{print int($2/1024)}\' /proc/meminfo)\n  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)\n  rss=$(echo "$line" | awk \'{print int($1/1024)}\')\n  proc=$(echo "$line" | awk \'{print $2}\')\n  echo "$epoch,$dt,$mem,$rss,$proc"\n  sleep 10\ndone\n"""\n    open("/tmp/mem_watch.sh", "w").write(watcher_sh)\n    os.chmod("/tmp/mem_watch.sh", 0o755)\n    p = subprocess.Popen(\n        ["/tmp/mem_watch.sh"],\n        stdout=open(f"{LOG_DIR}/mem-trace.csv", "w"),\n        stderr=subprocess.DEVNULL,\n    )\n    open("/tmp/watcher.pid", "w").write(str(p.pid))\n    print(f"[*] {now_hms()} mem-watcher pid={p.pid}")\n\n    print(f"[*] {now_hms()} installing tensorflow-tpu runtime")\n    run(\'export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax 2>&1 | tail -3\', check=False)\n    run(\n        \'export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system --quiet \'\n        "tensorflow-tpu==2.18.0 --find-links https://storage.googleapis.com/libtpu-tf-releases/index.html"\n    )\n\n    burner_py = r\'\'\'\nimport json, os, signal, sys, time, traceback\nos.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\nos.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")\nos.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nLOG_DIR = "/kaggle/working/logs"\nOK_FILE = f"{LOG_DIR}/tpu_burner.ok"\nJSONL = f"{LOG_DIR}/tpu_burner.jsonl"\nos.makedirs(LOG_DIR, exist_ok=True)\nstopping = False\ndef log(msg):\n    print("[tpu-burner " + time.strftime("%H:%M:%S") + "] " + str(msg), flush=True)\ndef write_ok(record):\n    tmp = OK_FILE + ".tmp"\n    with open(tmp, "w") as f:\n        json.dump(record, f, sort_keys=True)\n    os.replace(tmp, OK_FILE)\n    with open(JSONL, "a") as f:\n        f.write(json.dumps(record, sort_keys=True) + "\\n")\ndef term(signum, frame):\n    global stopping\n    stopping = True\n    log("signal received")\nsignal.signal(signal.SIGTERM, term)\nsignal.signal(signal.SIGINT, term)\ntry:\n    import tensorflow as tf\n    tf.config.threading.set_intra_op_parallelism_threads(1)\n    tf.config.threading.set_inter_op_parallelism_threads(1)\n    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")\n    log("initializing TPU")\n    resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")\n    tf.config.experimental_connect_to_cluster(resolver)\n    tf.tpu.experimental.initialize_tpu_system(resolver)\n    strategy = tf.distribute.TPUStrategy(resolver)\n    replicas = strategy.num_replicas_in_sync\n    log("TPU ready replicas=" + str(replicas))\n    dim, hidden, batch, steps_per_chunk = 1024, 2048, 1024, 8\n    with strategy.scope():\n        model = tf.keras.Sequential([\n            tf.keras.layers.InputLayer(shape=(dim,)),\n            tf.keras.layers.Dense(hidden, activation="gelu"),\n            tf.keras.layers.Dense(hidden, activation="gelu"),\n            tf.keras.layers.Dense(dim),\n        ])\n        opt = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-3)\n        x = tf.ones((batch, dim), dtype=tf.bfloat16)\n        target = tf.zeros((batch, dim), dtype=tf.bfloat16)\n        _ = model(x, training=True)\n        try:\n            opt.build(model.trainable_variables)\n        except AttributeError:\n            pass\n    @tf.function\n    def train_chunk():\n        total = tf.constant(0.0, tf.float32)\n        for _ in range(steps_per_chunk):\n            with tf.GradientTape() as tape:\n                pred = model(x, training=True)\n                diff = tf.cast(pred - target, tf.float32)\n                loss = tf.reduce_mean(diff * diff)\n            grads = tape.gradient(loss, model.trainable_variables)\n            opt.apply_gradients(zip(grads, model.trainable_variables))\n            total += loss\n        return total / tf.cast(steps_per_chunk, tf.float32)\n    chunk = 0\n    last_log = 0\n    while not stopping:\n        t0 = time.time()\n        per_replica = strategy.run(train_chunk)\n        reduced = strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica, axis=None)\n        elapsed = time.time() - t0\n        chunk += 1\n        record = {\n            "epoch": int(time.time()),\n            "datetime": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n            "chunk": chunk,\n            "replicas": int(replicas),\n            "steps_per_chunk": steps_per_chunk,\n            "elapsed_sec": round(elapsed, 3),\n            "loss": float(reduced.numpy()),\n        }\n        write_ok(record)\n        if time.time() - last_log >= 30:\n            log("chunk={chunk} loss={loss:.6f} elapsed={elapsed_sec:.2f}s replicas={replicas}".format(**record))\n            last_log = time.time()\n        time.sleep(0.2)\n    log("clean exit")\nexcept Exception as e:\n    log("FATAL " + repr(e))\n    traceback.print_exc()\n    sys.exit(2)\n\'\'\'\n    open(f"{LOG_DIR}/tpu_burner.py", "w").write(burner_py)\n\n    watchdog_py = r\'\'\'\nimport os, signal, subprocess, time\nLOG_DIR = "/kaggle/working/logs"\nBURNER = f"{LOG_DIR}/tpu_burner.py"\nOK_FILE = f"{LOG_DIR}/tpu_burner.ok"\nPID_FILE = "/tmp/tpu_burner.pid"\nBURNER_LOG = f"{LOG_DIR}/tpu_burner.log"\nWATCHDOG_LOG = f"{LOG_DIR}/tpu_watchdog.log"\nPY = "/usr/local/bin/python3"\nSTALE_SEC = 180\nFIRST_OK_STALE_SEC = 600\nCHECK_SEC = 60\ndef log(msg):\n    line = "[tpu-watchdog " + time.strftime("%H:%M:%S") + "] " + str(msg)\n    print(line, flush=True)\n    with open(WATCHDOG_LOG, "a") as f:\n        f.write(line + "\\n")\ndef read_pid():\n    try:\n        return int(open(PID_FILE).read().strip())\n    except Exception:\n        return None\ndef pid_alive(pid):\n    if not pid:\n        return False\n    try:\n        os.kill(pid, 0)\n        return True\n    except ProcessLookupError:\n        return False\n    except PermissionError:\n        return True\ndef stop_pid(pid):\n    if not pid_alive(pid):\n        return\n    try:\n        os.kill(pid, signal.SIGTERM)\n    except ProcessLookupError:\n        return\n    time.sleep(10)\n    if pid_alive(pid):\n        try:\n            os.kill(pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\ndef start_burner(reason):\n    old = read_pid()\n    if old:\n        stop_pid(old)\n    env = os.environ.copy()\n    env.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")\n    env.setdefault("TF_NUM_INTRAOP_THREADS", "1")\n    env.setdefault("TF_NUM_INTEROP_THREADS", "1")\n    env.setdefault("OMP_NUM_THREADS", "1")\n    logf = open(BURNER_LOG, "ab", buffering=0)\n    proc = subprocess.Popen([PY, BURNER], stdout=logf, stderr=subprocess.STDOUT, env=env, cwd="/kaggle/working")\n    open(PID_FILE, "w").write(str(proc.pid))\n    log("started burner pid={} reason={}".format(proc.pid, reason))\nstart_burner("initial")\nwhile True:\n    time.sleep(CHECK_SEC)\n    pid = read_pid()\n    age = None\n    if os.path.exists(OK_FILE):\n        age = time.time() - os.path.getmtime(OK_FILE)\n    if not pid_alive(pid):\n        start_burner("pid_dead")\n    elif age is None:\n        log("waiting for first ok pid={}".format(pid))\n        if os.path.exists(BURNER_LOG) and os.path.getsize(BURNER_LOG) > 0 and time.time() - os.path.getmtime(BURNER_LOG) > FIRST_OK_STALE_SEC:\n            start_burner("no_ok_stale_log")\n    elif age > STALE_SEC:\n        start_burner("ok_stale_{:.0f}s".format(age))\n    else:\n        log("healthy pid={} ok_age={:.0f}s".format(pid, age))\n\'\'\'\n    open(f"{LOG_DIR}/tpu_watchdog.py", "w").write(watchdog_py)\n    wd_log = open(f"{LOG_DIR}/tpu_watchdog.stdout", "ab", buffering=0)\n    wd = subprocess.Popen(\n        ["/usr/local/bin/python3", f"{LOG_DIR}/tpu_watchdog.py"],\n        stdout=wd_log,\n        stderr=subprocess.STDOUT,\n        cwd="/kaggle/working",\n    )\n    open("/tmp/tpu_watchdog.pid", "w").write(str(wd.pid))\n    print(f"[*] {now_hms()} tpu-watchdog pid={wd.pid}")\n    wait_for_tpu_burner(wd)\n\n\ndef wait_for_tpu_burner(wd_proc):\n    deadline = time.time() + 600\n    seen = set()\n    while time.time() < deadline:\n        if wd_proc.poll() is not None:\n            raise RuntimeError("TPU watchdog died early")\n        ok = f"{LOG_DIR}/tpu_burner.ok"\n        if os.path.exists(ok):\n            try:\n                rec = json.load(open(ok))\n                seen.add(int(rec.get("chunk", 0)))\n                if len(seen) >= 2:\n                    print(\n                        f"[*] {now_hms()} TPU burner verified chunks={sorted(seen)[-2:]} "\n                        f"elapsed={rec.get(\'elapsed_sec\')}s loss={rec.get(\'loss\')}"\n                    )\n                    run(f"tail -20 {LOG_DIR}/tpu_burner.log", check=False)\n                    run(f"tail -10 {LOG_DIR}/tpu_watchdog.log", check=False)\n                    return\n            except Exception as e:\n                print(f"[*] {now_hms()} waiting for TPU burner ok parse: {e}")\n        time.sleep(10)\n    run(f"tail -80 {LOG_DIR}/tpu_burner.log", check=False)\n    run(f"tail -80 {LOG_DIR}/tpu_watchdog.log", check=False)\n    raise RuntimeError("TPU burner did not produce two chunks within 10 minutes")\n\n\ndef run_phase2():\n    t0 = time.time()\n    os.chdir(os.environ["NPUCONVERT_DIR"])\n    run(\n        ". .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py",\n        log=f"{LOG_DIR}/phase2.log",\n    )\n    print(f"[*] {now_hms()} Phase2 elapsed={int(time.time() - t0)}s")\n\n\ndef run_phase3():\n    t0 = time.time()\n    os.chdir(os.environ["NPUCONVERT_DIR"])\n    run(\n        \'. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py --model_path "$MODEL_PATH"\',\n        log=f"{LOG_DIR}/phase3.log",\n    )\n    print(f"[*] {now_hms()} Phase3 elapsed={int(time.time() - t0)}s")\n    run(\'find . -name "*.onnx" -exec ls -lh {} \\\\;\', check=False)\n    run(\'find . -name "weights.pb" -exec ls -lh {} \\\\;\', check=False)\n\n\nRUN_STAGE = r\'\'\'\nstage_mark() {\n  local event="$1"; shift\n  local stage="$1"; shift\n  local cmd="${1:-}"\n  local ts="$(date -u +%Y-%m-%dT%H:%M:%SZ)"\n  echo "[[QNN_STAGE]] ts=${ts} event=${event} stage=${stage} cmd=${cmd} shell_pid=$$"\n  if [ -n "${QNN_STAGE_TSV:-}" ]; then\n    printf \'%s\\t%s\\t%s\\t%s\\t%s\\n\' "$ts" "$event" "$stage" "$cmd" "$$" >> "$QNN_STAGE_TSV"\n  fi\n  if [ -n "${QNN_CURRENT_STAGE:-}" ]; then\n    printf \'%s\\t%s\\t%s\\t%s\\t%s\\n\' "$ts" "$event" "$stage" "$cmd" "$$" > "$QNN_CURRENT_STAGE"\n  fi\n}\nrun_stage() {\n  local stage="$1"; shift\n  local cmd\n  printf -v cmd \'%q \' "$@"\n  stage_mark start "$stage" "$cmd"\n  set +e\n  "$@"\n  local rc=$?\n  set -e\n  stage_mark end "$stage" "$cmd rc=${rc}"\n  return "$rc"\n}\n\'\'\'\n\n\ndef patch_convert_scripts():\n    npu = os.environ["NPUCONVERT_DIR"]\n    qnn = os.environ["QNN_SDK_ROOT_DIR"]\n    soc = os.environ["MIN_SOC"]\n    for path in glob.glob(f"{npu}/scripts/convert_*.sh"):\n        orig = open(path).read()\n        s = re.sub(r"QNN_SDK_ROOT=/data/qairt/[0-9.]+", f"QNN_SDK_ROOT={qnn}", orig)\n        if "stage_mark()" not in s:\n            s = s.replace("set -e\\n", f"set -e\\ncd \\"{npu}\\"\\n{RUN_STAGE}\\n", 1)\n        name = os.path.basename(path).removeprefix("convert_").removesuffix("_sdxl.sh")\n        if not path.endswith("convert_all_sdxl.sh"):\n            out = []\n            cp_i = 0\n            for raw in s.splitlines():\n                stripped = raw.strip()\n                if stripped.startswith("qairt-converter "):\n                    raw = f\'run_stage "{name}/qairt-converter" {stripped}\'\n                elif stripped.startswith("qairt-quantizer "):\n                    raw = f\'run_stage "{name}/qairt-quantizer" {stripped}\'\n                elif stripped.startswith("qnn-context-binary-generator "):\n                    raw = f\'run_stage "{name}/qnn-context-binary-generator" {stripped}\'\n                elif stripped.startswith("./MNNConvert "):\n                    raw = f\'run_stage "{name}/MNNConvert" {stripped}\'\n                elif stripped.startswith("cp "):\n                    cp_i += 1\n                    raw = f\'run_stage "{name}/cp-{cp_i}" {stripped}\'\n                out.append(raw)\n            s = "\\n".join(out) + "\\n"\n        open(path, "w").write(s)\n        print(f"[*] {now_hms()} patched {os.path.basename(path)}")\n    hbp = f"{npu}/htp_backend_{soc}.json"\n    assert os.path.exists(hbp), f"{hbp} missing"\n    d = json.load(open(hbp))\n    d["backend_extensions"]["config_file_path"] = f"{npu}/htp_config_{soc}.json"\n    json.dump(d, open(hbp, "w"), indent=2)\n\n\nclass QnnSupervisor:\n    drop_keywords = ["Failed to set thread affinity for cpuset"]\n    print_keywords = [\n        "[[QNN_STAGE]]",\n        "INFO_CONVERSION_SUCCESS",\n        "Quantized Model saved at",\n        "INFO_WRITE_SUCCESS",\n        "Converted Success!",\n        "Model larger than 2GB",\n        "Elapsed (wall clock)",\n        "Maximum resident set size",\n        "ERROR",\n        "Error",\n        "error:",\n        "Traceback",\n        "Cannot open",\n        "not found",\n        "No such file",\n        "Killed",\n        "Aborted",\n        "Segmentation fault",\n        "context-binary",\n        "qnn-context-binary-generator",\n        "qairt-quantizer",\n        "qairt-converter",\n    ]\n\n    def __init__(self, label):\n        self.label = label\n        self.npu = os.environ["NPUCONVERT_DIR"]\n        self.soc = os.environ["MIN_SOC"]\n        self.phase_log = f"{LOG_DIR}/{label}.log"\n        self.affinity_log = f"{LOG_DIR}/{label}.affinity.sample.log"\n        self.stage_tsv = f"{LOG_DIR}/{label}.qnn_stage.tsv"\n        self.current_stage_file = f"{LOG_DIR}/{label}.qnn_current_stage.tsv"\n        self.status_tsv = f"{LOG_DIR}/{label}.qnn_status.tsv"\n        self.artifact_tsv = f"{LOG_DIR}/{label}.qnn_artifacts.tsv"\n        self.progress_tsv = f"{LOG_DIR}/{label}.qnn_progress.tsv"\n        self.last_milestone = "starting"\n        self.suppressed_affinity = 0\n        self.printed_lines = 0\n        self.exec_start = 0\n        self.exec_end = 0\n        self.last_status_exec_end = 0\n        self.stage_input_cache = {}\n        for p in [self.phase_log, self.affinity_log, self.stage_tsv, self.current_stage_file]:\n            open(p, "w").close()\n        open(self.status_tsv, "w").write(\n            "epoch\\tts\\telapsed_s\\tmem_avail_mb\\tdisk_avail_mb\\tphase_log_mib\\tstage\\tproc\\ttpu\\t"\n            "suppressed_affinity\\taffinity_rate_per_min\\twarnings\\tlast_signal\\n"\n        )\n        open(self.artifact_tsv, "w").write("epoch\\tts\\tstage\\tout_files\\tout_mib\\tkey_files\\n")\n        open(self.progress_tsv, "w").write(\n            "epoch\\tts\\tstage\\tstage_age_s\\tinput_list\\tinput_rows\\texec_start\\texec_end\\t"\n            "exec_delta_per_min\\tpass_est\\tlast_qnn_ms\\n"\n        )\n\n    def dropped_count(self, line):\n        total = 0\n        for keyword in self.drop_keywords:\n            if keyword in line:\n                total += max(1, line.count(keyword))\n        if total == 0 and re.fullmatch(r"(WARNING:\\s*)+", line.strip()):\n            total = 1\n        return total\n\n    def handle_line(self, line, logf, afff):\n        clean = line.rstrip()\n        if "[QNN_CPU] QnnGraph execute start" in clean:\n            self.exec_start += 1\n        if "[QNN_CPU] QnnGraph execute end" in clean:\n            self.exec_end += 1\n        drop_n = self.dropped_count(clean)\n        if drop_n:\n            before = self.suppressed_affinity\n            self.suppressed_affinity += drop_n\n            if before < 20 or self.suppressed_affinity // 100000 != before // 100000:\n                afff.write(f"{now_hms()} count={self.suppressed_affinity} delta={drop_n} {clean[:1000]}\\n")\n                afff.flush()\n            return\n        logf.write(line)\n        logf.flush()\n        if clean:\n            self.last_milestone = clean[-240:]\n        if any(k in clean for k in self.print_keywords):\n            print(clean[:1200], flush=True)\n            self.printed_lines += 1\n\n    def tpu_status(self):\n        ok = f"{LOG_DIR}/tpu_burner.ok"\n        if not os.path.exists(ok):\n            return "missing"\n        age = time.time() - os.path.getmtime(ok)\n        try:\n            rec = json.load(open(ok))\n            return f"age={int(age)}s chunk={rec.get(\'chunk\')} elapsed={rec.get(\'elapsed_sec\')}"\n        except Exception:\n            return f"age={int(age)}s parse_error"\n\n    def tail_line(self, path):\n        try:\n            with open(path, "rb") as f:\n                f.seek(0, os.SEEK_END)\n                size = f.tell()\n                f.seek(max(0, size - 4096))\n                lines = f.read().decode("utf-8", "replace").splitlines()\n                return lines[-1] if lines else ""\n        except Exception:\n            return ""\n\n    def current_stage(self):\n        try:\n            raw = open(self.current_stage_file).read().strip()\n            if not raw:\n                return "missing", None, None, None, ""\n            parts = raw.split("\\t")\n            ts, event, stage, cmd = (parts + ["?", "?", "?", "?"])[:4]\n            age = time.time() - os.path.getmtime(self.current_stage_file)\n            return f"{event}:{stage} age={int(age)}s cmd={cmd}", event, stage, age, cmd\n        except Exception as e:\n            return f"error:{e}", None, None, None, ""\n\n    def qnn_proc_stats(self):\n        keys = ("qnn-context-binary-generator", "qairt-quantizer", "qairt-converter", "MNNConvert")\n        try:\n            out = subprocess.check_output(\n                ["ps", "-eo", "pid,ppid,etimes,pcpu,rss,vsz,comm,args"],\n                text=True,\n                stderr=subprocess.DEVNULL,\n            )\n        except Exception as e:\n            return f"ps_error:{e}"\n        rows = []\n        for line in out.splitlines()[1:]:\n            if not any(k in line for k in keys):\n                continue\n            parts = line.split(None, 7)\n            if len(parts) < 8:\n                continue\n            pid, _ppid, etimes, pcpu, rss, vsz, comm, _args = parts\n            try:\n                rss_mib = int(rss) // 1024\n                vsz_mib = int(vsz) // 1024\n            except ValueError:\n                rss_mib = vsz_mib = -1\n            rows.append(f"{comm} pid={pid} age={etimes}s cpu={pcpu}% rss={rss_mib}MiB vsz={vsz_mib}MiB")\n        return "; ".join(rows[:4]) if rows else "none"\n\n    def burner_proc_stats(self):\n        try:\n            out = subprocess.check_output(["ps", "-eo", "pid,etimes,pcpu,rss,comm,args"], text=True)\n        except Exception:\n            return "none"\n        rows = []\n        for line in out.splitlines()[1:]:\n            if "tpu_burner.py" not in line and "tpu_watchdog.py" not in line:\n                continue\n            parts = line.split(None, 5)\n            if len(parts) < 6:\n                continue\n            pid, etimes, pcpu, rss, comm, _args = parts\n            rows.append(f"{comm} pid={pid} age={etimes}s cpu={pcpu}% rss={int(rss)//1024}MiB")\n        return "; ".join(rows) if rows else "none"\n\n    def artifact_summary(self):\n        out_dir = f"{self.npu}/output/qnn_models_sdxl_{self.soc}"\n        files, total = 0, 0\n        if os.path.isdir(out_dir):\n            for root, _, names in os.walk(out_dir):\n                for name in names:\n                    p = os.path.join(root, name)\n                    try:\n                        files += 1\n                        total += os.path.getsize(p)\n                    except OSError:\n                        pass\n        keys = []\n        for pat in [f"{self.npu}/*_sdxl/model.dlc", f"{self.npu}/*_sdxl/model_quantized.dlc", f"{out_dir}/*"]:\n            for p in sorted(glob.glob(pat)):\n                if os.path.isfile(p):\n                    keys.append(f"{os.path.relpath(p, self.npu)}:{file_mib(p):.1f}MiB")\n        return files, total / (1024 * 1024), ";".join(keys[-30:])\n\n    def input_list_summary(self, stage, cmd):\n        m = re.search(r"--input_list\\s+(\\S+)", cmd or "")\n        if not m:\n            return "", 0\n        raw = m.group(1).strip("\'\\"")\n        path = raw if os.path.isabs(raw) else os.path.join(self.npu, raw)\n        if path in self.stage_input_cache:\n            return self.stage_input_cache[path]\n        rows = 0\n        try:\n            for line in open(path):\n                s = line.strip()\n                if s and not s.startswith("#"):\n                    rows += 1\n        except Exception:\n            rows = 0\n        self.stage_input_cache[path] = (raw, rows)\n        return raw, rows\n\n    def progress_record(self, stage_status, stage_age, cmd, now, last_affinity_time):\n        input_list, rows = self.input_list_summary(stage_status, cmd)\n        exec_delta = self.exec_end - self.last_status_exec_end\n        self.last_status_exec_end = self.exec_end\n        exec_rate = 60.0 * exec_delta / max(1.0, now - last_affinity_time)\n        pass_est = (self.exec_end / rows) if rows else 0.0\n        last_ms = ""\n        m = re.search(r"([0-9.]+)ms", self.last_milestone)\n        if m:\n            last_ms = m.group(1)\n        with open(self.progress_tsv, "a") as pf:\n            pf.write(\n                "\\t".join(\n                    map(\n                        tsv_escape,\n                        [\n                            int(now),\n                            now_hms(),\n                            stage_status,\n                            int(stage_age or 0),\n                            input_list,\n                            rows,\n                            self.exec_start,\n                            self.exec_end,\n                            f"{exec_rate:.2f}",\n                            f"{pass_est:.3f}",\n                            last_ms,\n                        ],\n                    )\n                )\n                + "\\n"\n            )\n        return f"input_rows={rows} exec={self.exec_end}/{self.exec_start} pass_est={pass_est:.2f} exec_rate={exec_rate:.1f}/min"\n\n    def run(self, command, required_outputs=None):\n        os.environ["QNN_STAGE_TSV"] = self.stage_tsv\n        os.environ["QNN_CURRENT_STAGE"] = self.current_stage_file\n        env = os.environ.copy()\n        for k in ["TF_NUM_INTRAOP_THREADS", "TF_NUM_INTEROP_THREADS", "OMP_NUM_THREADS"]:\n            env.pop(k, None)\n        print(f"[*] {now_hms()} QNN supervisor start label={self.label}")\n        proc = subprocess.Popen(\n            ["bash", "-lc", "set -o pipefail; " + command],\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n            cwd=self.npu,\n            env=env,\n        )\n        t0 = time.time()\n        last_status = 0.0\n        last_affinity_count = 0\n        last_affinity_time = time.time()\n        with open(self.phase_log, "a", buffering=1) as logf, open(self.affinity_log, "a", buffering=1) as afff:\n            while True:\n                ready, _, _ = select.select([proc.stdout], [], [], 1)\n                if ready:\n                    line = proc.stdout.readline()\n                    if line:\n                        self.handle_line(line, logf, afff)\n                    elif proc.poll() is not None:\n                        break\n                if proc.poll() is not None:\n                    for line in proc.stdout:\n                        self.handle_line(line, logf, afff)\n                    break\n                now = time.time()\n                if now - last_status >= 60:\n                    stage_status, _stage_event, _stage_name, stage_age, cmd = self.current_stage()\n                    proc_status = self.qnn_proc_stats()\n                    phase_mib = file_mib(self.phase_log)\n                    affinity_rate = 60.0 * (self.suppressed_affinity - last_affinity_count) / max(\n                        1.0, now - last_affinity_time\n                    )\n                    progress = self.progress_record(stage_status, stage_age, cmd, now, last_affinity_time)\n                    last_affinity_count = self.suppressed_affinity\n                    last_affinity_time = now\n                    files, out_mib, key_files = self.artifact_summary()\n                    warn = []\n                    if disk_avail_mb() < 4096:\n                        warn.append("low_disk_free")\n                    warn_text = ",".join(warn) if warn else "none"\n                    wd_tail = self.tail_line(f"{LOG_DIR}/tpu_watchdog.log")[-180:]\n                    msg = (\n                        f"[*] {now_hms()} QNN running label={self.label} elapsed={int(now - t0)}s "\n                        f"log={phase_mib:.1f}MiB disk_free={disk_avail_mb()}MiB mem_avail={mem_avail_mb()}MiB "\n                        f"tpu_ok={self.tpu_status()} stage=\\"{stage_status}\\" proc=\\"{proc_status}\\" "\n                        f"burner=\\"{self.burner_proc_stats()}\\" progress=\\"{progress}\\" "\n                        f"artifacts=\\"files={files} mib={out_mib:.1f} keys={key_files[-700:]}\\" "\n                        f"suppressed_affinity={self.suppressed_affinity} affinity_rate={affinity_rate:.0f}/min "\n                        f"warnings={warn_text} last_signal=\\"{self.last_milestone[-200:]}\\" watchdog=\\"{wd_tail}\\""\n                    )\n                    print(msg.replace(\'"\', "\'"), flush=True)\n                    with open(self.status_tsv, "a") as sf:\n                        sf.write(\n                            "\\t".join(\n                                map(\n                                    tsv_escape,\n                                    [\n                                        int(now),\n                                        now_hms(),\n                                        int(now - t0),\n                                        mem_avail_mb(),\n                                        disk_avail_mb(),\n                                        f"{phase_mib:.1f}",\n                                        stage_status,\n                                        proc_status,\n                                        self.tpu_status(),\n                                        self.suppressed_affinity,\n                                        f"{affinity_rate:.1f}",\n                                        warn_text,\n                                        self.last_milestone,\n                                    ],\n                                )\n                            )\n                            + "\\n"\n                        )\n                    with open(self.artifact_tsv, "a") as af:\n                        af.write(\n                            "\\t".join(\n                                map(\n                                    tsv_escape,\n                                    [int(now), now_hms(), stage_status, files, f"{out_mib:.1f}", key_files],\n                                )\n                            )\n                            + "\\n"\n                        )\n                    last_status = now\n        rc = proc.wait()\n        print(\n            f"[*] {now_hms()} QNN supervisor exit label={self.label} rc={rc} "\n            f"printed={self.printed_lines} suppressed_affinity={self.suppressed_affinity} "\n            f"exec_start={self.exec_start} exec_end={self.exec_end}"\n        )\n        if rc != 0:\n            run(f"tail -120 {self.phase_log}", check=False)\n            raise RuntimeError(f"QNN command failed rc={rc}: {command[:240]}")\n        if required_outputs:\n            missing = [p for p in required_outputs if not os.path.exists(p)]\n            if missing:\n                raise RuntimeError("QNN command completed but required outputs missing:\\n" + "\\n".join(missing))\n\n\ndef prepare_qnn_runtime():\n    npu = os.environ["NPUCONVERT_DIR"]\n    pylib = subprocess.check_output(\n        [".venv/bin/python", "-c", \'import os,sys; print(os.path.join(sys.base_prefix, "lib"))\'],\n        cwd=npu,\n        text=True,\n    ).strip()\n    assert os.path.exists(os.path.join(pylib, "libpython3.10.so.1.0")), f"libpython missing in {pylib}"\n    os.environ["LD_LIBRARY_PATH"] = f"{pylib}:{os.environ.get(\'LD_LIBRARY_PATH\', \'\')}"\n    qnn_lib = f"{os.environ[\'QNN_SDK_ROOT_DIR\']}/lib/x86_64-linux-clang"\n    os.environ["LD_LIBRARY_PATH"] = f"{qnn_lib}:{os.environ[\'LD_LIBRARY_PATH\']}"\n    print(f"[*] {now_hms()} LD_LIBRARY_PATH ready")\n\n\ndef qnn_driver_command(scripts):\n    npu = os.environ["NPUCONVERT_DIR"]\n    qnn = os.environ["QNN_SDK_ROOT_DIR"]\n    qnn_bin = os.environ["QNN_SDK_BIN"]\n    lines = [\n        f\'cd "{npu}"\',\n        f\'export QNN_SDK_ROOT="{qnn}"\',\n        f\'source "{qnn_bin}/envsetup.sh"\',\n    ]\n    for script in scripts:\n        lines.append(f\'bash "scripts/{script}" --min_soc "$MIN_SOC"\')\n    return " && ".join(lines)\n\n\ndef referenced_input_files(input_list_path, base_dir):\n    deps = []\n    rows = 0\n    missing = []\n    for line in open(input_list_path):\n        s = line.strip()\n        if not s or s.startswith("#"):\n            continue\n        rows += 1\n        for tok in re.split(r"\\s+", s):\n            val = tok\n            if ":=" in val:\n                val = val.split(":=", 1)[1]\n            elif "=" in val:\n                val = val.split("=", 1)[1]\n            val = val.strip("\'\\"")\n            if not val or val.startswith("[") or val.startswith("{"):\n                continue\n            path = val if os.path.isabs(val) else os.path.join(base_dir, val)\n            if os.path.isfile(path):\n                deps.append(os.path.abspath(path))\n            elif "/" in val or "." in os.path.basename(val):\n                missing.append(val)\n    return rows, sorted(set(deps)), missing\n\n\ndef copy_preserve(src, dst_root, base_dir):\n    src_abs = os.path.abspath(src)\n    try:\n        rel = os.path.relpath(src_abs, base_dir)\n        if rel.startswith(".."):\n            rel = os.path.join("_external", os.path.basename(src_abs))\n    except ValueError:\n        rel = os.path.join("_external", os.path.basename(src_abs))\n    dst = os.path.join(dst_root, rel)\n    os.makedirs(os.path.dirname(dst), exist_ok=True)\n    shutil.copy2(src_abs, dst)\n    return rel, os.path.getsize(src_abs)\n\n\ndef create_b1_bundle():\n    npu = os.environ["NPUCONVERT_DIR"]\n    soc = os.environ["MIN_SOC"]\n    bundle = "/kaggle/working/b1_bundle"\n    if os.path.exists(bundle):\n        shutil.rmtree(bundle)\n    os.makedirs(bundle)\n    output_dir = f"{npu}/output/qnn_models_sdxl_{soc}"\n    assert os.path.isdir(output_dir), f"{output_dir} missing"\n    shutil.copytree(output_dir, f"{bundle}/output/qnn_models_sdxl_{soc}")\n    for rel in [\n        "unet_sdxl/model.onnx",\n        "input_list_unet_sdxl.txt",\n        f"htp_backend_{soc}.json",\n        f"htp_config_{soc}.json",\n        "tokenizer.json",\n    ]:\n        src = f"{npu}/{rel}"\n        assert os.path.exists(src), f"required B1 bundle file missing: {src}"\n        os.makedirs(os.path.dirname(f"{bundle}/{rel}"), exist_ok=True)\n        shutil.copy2(src, f"{bundle}/{rel}")\n    rows, deps, missing = referenced_input_files(f"{npu}/input_list_unet_sdxl.txt", npu)\n    dep_bytes = 0\n    dep_rels = []\n    for dep in deps:\n        rel, size = copy_preserve(dep, bundle, npu)\n        dep_bytes += size\n        dep_rels.append(rel)\n    manifest = {\n        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n        "model_name": os.environ["MODEL_NAME"],\n        "min_soc": soc,\n        "input_list_rows": rows,\n        "dependency_count": len(dep_rels),\n        "dependency_mib": round(dep_bytes / (1024 * 1024), 1),\n        "dependencies": dep_rels[:2000],\n        "missing_tokens": missing[:200],\n    }\n    json.dump(manifest, open(f"{bundle}/manifest.json", "w"), indent=2)\n    print(f"[*] {now_hms()} B1 bundle manifest: {json.dumps(manifest)[:1200]}")\n    run(f"du -sh {bundle}/*", check=False)\n\n\ndef locate_b1_bundle():\n    hits = glob.glob("/kaggle/input/**/b1_bundle/manifest.json", recursive=True)\n    assert hits, "b1_bundle/manifest.json not found in /kaggle/input"\n    bundle = os.path.dirname(hits[0])\n    manifest = json.load(open(hits[0]))\n    print(f"[*] {now_hms()} B1 bundle={bundle} manifest={json.dumps(manifest)[:1200]}")\n    return bundle, manifest\n\n\ndef restore_b1_bundle_to_npu():\n    npu = os.environ["NPUCONVERT_DIR"]\n    bundle, manifest = locate_b1_bundle()\n    for name in os.listdir(bundle):\n        src = os.path.join(bundle, name)\n        dst = os.path.join(npu, name)\n        if os.path.isdir(src):\n            if os.path.exists(dst):\n                shutil.rmtree(dst)\n            shutil.copytree(src, dst)\n        else:\n            shutil.copy2(src, dst)\n    soc = os.environ["MIN_SOC"]\n    required = [\n        f"{npu}/unet_sdxl/model.onnx",\n        f"{npu}/input_list_unet_sdxl.txt",\n        f"{npu}/htp_backend_{soc}.json",\n        f"{npu}/htp_config_{soc}.json",\n        f"{npu}/output/qnn_models_sdxl_{soc}/vae_encoder.bin",\n        f"{npu}/output/qnn_models_sdxl_{soc}/vae_decoder.bin",\n    ]\n    missing = [p for p in required if not os.path.exists(p)]\n    assert not missing, "B1 restore missing files:\\n" + "\\n".join(missing)\n    rows, deps, missing_tokens = referenced_input_files(f"{npu}/input_list_unet_sdxl.txt", npu)\n    if missing_tokens:\n        print(f"[*] {now_hms()} input_list unresolved tokens sample={missing_tokens[:20]}")\n    print(f"[*] {now_hms()} restored B1 bundle rows={rows} deps={len(deps)}")\n    return manifest\n\n\ndef package_final_zip():\n    npu = os.environ["NPUCONVERT_DIR"]\n    soc = os.environ["MIN_SOC"]\n    out_dir = f"{npu}/output/qnn_models_sdxl_{soc}"\n    assert os.path.isdir(out_dir), f"{out_dir} missing"\n    assert glob.glob(f"{out_dir}/unet*"), "UNet output not found in final output dir"\n    run(f\'touch "{out_dir}/SDXL"\')\n    zip_name = f"{os.environ[\'MODEL_NAME\']}_qnn2.28_{soc}.zip"\n    run("df -h /kaggle/working", check=False)\n    run(f\'zip -r "/kaggle/working/{zip_name}" "{out_dir}" > "{LOG_DIR}/package_zip.log"\')\n    run(f\'ls -lh "/kaggle/working/{zip_name}"\')\n    run("df -h /kaggle/working", check=False)\n\n\ndef stop_background_services():\n    for pidfile, name in [\n        ("/tmp/tpu_watchdog.pid", "tpu-watchdog"),\n        ("/tmp/tpu_burner.pid", "tpu-burner"),\n        ("/tmp/watcher.pid", "mem-watcher"),\n    ]:\n        try:\n            pid = int(open(pidfile).read().strip())\n        except Exception:\n            continue\n        try:\n            os.kill(pid, signal.SIGTERM)\n            print(f"[*] {now_hms()} SIGTERM {name} pid={pid}")\n        except ProcessLookupError:\n            continue\n\n\ndef common_bootstrap(model_name, min_soc, realistic, civitai_version_id=None):\n    configure_env(model_name, min_soc, realistic, civitai_version_id)\n    run("free -h", check=False)\n    install_tools_and_convert_package()\n    install_qnn_sdk()\n    setup_python_env()\n    start_background_services()\n\n\ndef run_b1(model_name, min_soc, realistic, civitai_version_id=None):\n    common_bootstrap(model_name, min_soc, realistic, civitai_version_id)\n    locate_phase1_output()\n    copy_phase1_calibration_data()\n    run_phase2()\n    run_phase3()\n    patch_convert_scripts()\n    prepare_qnn_runtime()\n    npu = os.environ["NPUCONVERT_DIR"]\n    soc = os.environ["MIN_SOC"]\n    required = [\n        f"{npu}/output/qnn_models_sdxl_{soc}/clip.mnn",\n        f"{npu}/output/qnn_models_sdxl_{soc}/clip_2.mnn",\n        f"{npu}/output/qnn_models_sdxl_{soc}/clip_2.mnn.weight",\n        f"{npu}/output/qnn_models_sdxl_{soc}/vae_encoder.bin",\n        f"{npu}/output/qnn_models_sdxl_{soc}/vae_decoder.bin",\n    ]\n    scripts = [\n        "convert_clip_sdxl.sh",\n        "convert_clip2_sdxl.sh",\n        "convert_vae_encoder_sdxl.sh",\n        "convert_vae_decoder_sdxl.sh",\n    ]\n    QnnSupervisor("b1_phase45").run(qnn_driver_command(scripts), required_outputs=required)\n    create_b1_bundle()\n    stop_background_services()\n    print(f"[*] {now_hms()} B1 done")\n\n\ndef run_b2(model_name, min_soc, realistic, civitai_version_id=None):\n    common_bootstrap(model_name, min_soc, realistic, civitai_version_id)\n    restore_b1_bundle_to_npu()\n    patch_convert_scripts()\n    prepare_qnn_runtime()\n    npu = os.environ["NPUCONVERT_DIR"]\n    soc = os.environ["MIN_SOC"]\n    required = [f"{npu}/output/qnn_models_sdxl_{soc}/unet.bin"]\n    QnnSupervisor("b2_unet").run(qnn_driver_command(["convert_unet_sdxl.sh"]), required_outputs=required)\n    package_final_zip()\n    stop_background_services()\n    print(f"[*] {now_hms()} B2 done")\n'
exec(COMMON_CODE, globals())
run_b2(MODEL_NAME, MIN_SOC, REALISTIC, CIVITAI_VERSION_ID)
